In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
Leaky RND + Policy Gradient agent for ARC-AGI-3.

Intrinsic reward:  r^i = 1/2 * ||P(phi(s)) - T(phi(s))||^2
Leak update:       theta_P <- (1-mu)*theta_P + mu*theta_P_init   (per update)

phi = frozen-random linear projection of CNN features (stable from step 0, no ICM needed).
Policy = REINFORCE with baseline (value head). Updates every UPDATE_EVERY steps.
"""
import random
import time
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


# ── constants ─────────────────────────────────────────────────────────────────
GRID     = 64
N_COLORS = 16
# Map raw int available_actions (1-7) to GameAction enums
ACTION_MAP = {i: getattr(GameAction, f'ACTION{i}') for i in range(1, 8)}
ACTION_MAP[0] = GameAction.RESET
N_SLOTS = 8   # 0=RESET, 1-7=ACTION1-7


# ── CNN encoder ───────────────────────────────────────────────────────────────
class CNNEncoder(nn.Module):
    def __init__(self, in_ch=N_COLORS, feat_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32,  3, padding=1), nn.ReLU(),
            nn.Conv2d(32,   64,  3, padding=1), nn.ReLU(),
            nn.Conv2d(64,   128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128,  256, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, feat_dim), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)


# ── Leaky RND  (theta_P <- (1-mu)*theta_P + mu*theta_P_init) ──────────────────
class _MLP(nn.Module):
    def __init__(self, in_d, hid, out_d, n=2):
        super().__init__()
        seq = [nn.Linear(in_d, hid), nn.ReLU()]
        for _ in range(n - 1): seq += [nn.Linear(hid, hid), nn.ReLU()]
        seq.append(nn.Linear(hid, out_d))
        self.net = nn.Sequential(*seq)
    def forward(self, x): return self.net(x)

class LeakyRND(nn.Module):
    def __init__(self, phi_dim=128, hid=256, out=256, leak=0.05):
        super().__init__()
        self.target    = _MLP(phi_dim, hid, out, n=2)
        self.predictor = _MLP(phi_dim, hid, out, n=3)
        for p in self.target.parameters(): p.requires_grad_(False)
        self.target.eval()
        self.leak = leak
        self._init = [p.detach().clone() for p in self.predictor.parameters()]

    @torch.no_grad()
    def novelty(self, phi):
        return 0.5 * (self.predictor(phi) - self.target(phi)).pow(2).mean(-1)

    def distill_loss(self, phi):
        with torch.no_grad(): t = self.target(phi)
        return (self.predictor(phi) - t).pow(2).mean()

    @torch.no_grad()
    def apply_leak(self):
        for p, p0 in zip(self.predictor.parameters(), self._init):
            p.mul_(1 - self.leak).add_(p0.to(p.device), alpha=self.leak)


# ── Frozen random projection phi = W * feat ───────────────────────────────────
class FrozenProj(nn.Module):
    def __init__(self, in_d=256, out_d=128, seed=42):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.register_buffer('W', torch.randn(in_d, out_d, generator=g) / in_d**0.5)
    def forward(self, x): return x @ self.W


# ── Policy head (action logits + value + ACTION6 coords) ─────────────────────
class PolicyHead(nn.Module):
    def __init__(self, feat_dim=256):
        super().__init__()
        self.action = nn.Linear(feat_dim, N_SLOTS)
        self.value  = nn.Linear(feat_dim, 1)
        self.x_head = nn.Linear(feat_dim, GRID)
        self.x_emb  = nn.Embedding(GRID, 32)
        self.y_head = nn.Linear(feat_dim + 32, GRID)

    def sample(self, feat, avail_slots):
        logits = self.action(feat)
        mask   = torch.full_like(logits, float('-inf'))
        mask[:, avail_slots] = 0.0
        dist   = torch.distributions.Categorical(logits=logits + mask)
        slot   = dist.sample()
        logp   = dist.log_prob(slot)
        v      = self.value(feat).squeeze(-1)
        x = y  = None
        if int(slot) == 6:
            xd = torch.distributions.Categorical(logits=self.x_head(feat))
            x  = xd.sample()
            yd = torch.distributions.Categorical(
                    logits=self.y_head(torch.cat([feat, self.x_emb(x)], dim=-1)))
            y  = yd.sample()
            logp = logp + xd.log_prob(x) + yd.log_prob(y)
        return int(slot), x, y, logp, v


# ── Agent ─────────────────────────────────────────────────────────────────────
class LeakyRNDAgent(Agent):
    MAX_ACTIONS = float('inf')

    # hyper-parameters
    FEAT_DIM     = 256
    PHI_DIM      = 128
    LEAK         = 0.05
    LR           = 3e-4
    UPDATE_EVERY = 32
    GAMMA        = 0.99
    ENT_COEF     = 0.05
    NORM_BETA    = 0.99
    NORM_EPS     = 1e-4
    CLIP         = 5.0

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)   # sets self.game_id
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.encoder = CNNEncoder(N_COLORS, self.FEAT_DIM).to(self.device)
        self.phi_proj = FrozenProj(self.FEAT_DIM, self.PHI_DIM).to(self.device)
        self.rnd      = LeakyRND(self.PHI_DIM, 256, 256, self.LEAK).to(self.device)
        self.policy   = PolicyHead(self.FEAT_DIM).to(self.device)
        for p in self.phi_proj.parameters(): p.requires_grad_(False)

        self.opt = torch.optim.Adam(
            list(self.encoder.parameters()) +
            list(self.rnd.predictor.parameters()) +
            list(self.policy.parameters()), lr=self.LR)

        self._nov_ema = 1.0
        self._buf: list[dict] = []
        self._step  = 0
        self._level = 0
        print(f'[LeakyRND] game={self.game_id}  device={self.device}', flush=True)

    # ── required ──────────────────────────────────────────────────────────────

    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        # level advance
        lvl = int(getattr(latest_frame, 'levels_completed', 0) or 0)
        if lvl > self._level:
            print(f'[LeakyRND] level {self._level}->{lvl} @ step {self._step}', flush=True)
            self._buf.clear()
            self._level = lvl

        # non-playing states
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET

        # available actions -> slot indices (raw ints in available_actions)
        avail_raw = list(getattr(latest_frame, 'available_actions', [1]) or [1])
        avail_slots = list({0} | {int(a) for a in avail_raw if 0 <= int(a) < N_SLOTS})

        # forward
        obs  = self._obs(latest_frame)
        with torch.no_grad():
            feat = self.encoder(obs)
            phi  = self.phi_proj(feat)
            nov  = float(self.rnd.novelty(phi).item())

        # normalise novelty
        self._nov_ema = self.NORM_BETA * self._nov_ema + (1 - self.NORM_BETA) * nov**2
        r_int = float(np.clip(nov / (self._nov_ema**0.5 + self.NORM_EPS),
                               -self.CLIP, self.CLIP))

        slot, x, y, logp, v = self.policy.sample(feat, avail_slots)

        self._buf.append({'logp': logp.detach(), 'value': v.detach(),
                          'r_int': r_int, 'phi': phi.detach(),
                          'feat': feat.detach()})
        self._step += 1
        if self._step % self.UPDATE_EVERY == 0 and len(self._buf) >= 4:
            self._update()

        # build GameAction
        if slot == 0:
            action = GameAction.RESET
        else:
            action = ACTION_MAP.get(slot, GameAction.ACTION1)

        if action.is_simple():
            action.reasoning = f'leaky-rnd step={self._step} nov={r_int:.3f}'
        elif action.is_complex():
            cx, cy = (int(x), int(y)) if x is not None else (32, 32)
            action.set_data({'x': cx, 'y': cy})
            action.reasoning = {'x': cx, 'y': cy, 'nov': round(r_int, 3)}
        return action

    # ── update ────────────────────────────────────────────────────────────────

    def _update(self):
        buf = self._buf[-self.UPDATE_EVERY:]
        logps  = torch.stack([t['logp']              for t in buf])
        values = torch.stack([t['value'].squeeze(-1) for t in buf])
        phis   = torch.cat([t['phi']   for t in buf], dim=0)
        feats  = torch.cat([t['feat']  for t in buf], dim=0)

        G, returns = 0.0, []
        for r in reversed([t['r_int'] for t in buf]):
            G = r + self.GAMMA * G; returns.insert(0, G)
        ret = torch.tensor(returns, dtype=torch.float32, device=self.device)
        ret = (ret - ret.mean()) / (ret.std() + 1e-8)

        adv     = ret - values.detach()
        pg_loss = -(logps * adv).mean()
        v_loss  = F.mse_loss(values, ret)
        entropy = torch.distributions.Categorical(
                      logits=self.policy.action(feats)).entropy().mean()
        rnd_loss = self.rnd.distill_loss(phis)

        loss = pg_loss + 0.5 * v_loss - self.ENT_COEF * entropy + rnd_loss
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(self.encoder.parameters()) + list(self.policy.parameters()), 0.5)
        self.opt.step()
        self.rnd.apply_leak()
        self._buf.clear()

    # ── helper ────────────────────────────────────────────────────────────────

    def _obs(self, fd):
        frame = np.array(fd.frame, dtype=np.int64)[-1]
        t = torch.zeros(N_COLORS, GRID, GRID, dtype=torch.float32)
        t.scatter_(0, torch.from_numpy(frame).unsqueeze(0), 1.0)
        return t.unsqueeze(0).to(self.device)

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the competition-provided ARC-AGI-3-Agents repo to a writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Place our agent in the templates directory
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register our agent (overwrite __init__.py to avoid importing heavy deps)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import LeakyRNDAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random":   Random,
    "leakyrnd": LeakyRNDAgent,
}
""")

    # Write .env for online mode
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run the agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent leakyrnd

In [ ]:
# Non-rerun: produce a dummy submission so Kaggle accepts the notebook
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Dummy submission written (non-rerun mode).')